# 01 · Data sourcing

Three sources, all limited to information available before **12am Sep 17, 2026 (Eastern)**:

| Source | What it is | Used for | Saved to |
|---|---|---|---|
| NOAA GHCNh, station USW00013722 (RDU airport) | measured hourly weather at the airport, Jan 2015 – Sep 16, 2026 | the target (temperature), climatology, recent anomalies | `data/raw/<year>/rdu_ghcnh_<first day>_to_<last day>.csv` (one file per Eastern year) and the merged `data/raw/rdu_ghcnh_2015-01-01_to_2026-09-16.csv` |
| ECMWF IFS, via the Open-Meteo Single Runs API | archived runs of the European Centre's physics-based global weather model: 15-day hourly forecasts, issued twice a day | forecast inputs for the linear regression (MOS) | `data/raw/ecmwf_ifs_runs_2024-03-14_to_2026-09-16.csv` |
| NOAA GFS (`gfs_seamless`), via the same API | archived 12z runs of the US weather service's global model, from Apr 2, 2026 (start of the archive) | comparison only: a second raw forecast next to ECMWF in `04` and `05` | `data/raw/gfs_seamless_runs_2026-04-02_to_2026-09-16.csv` |

## 1. Settings

In [14]:
import json
import time
import urllib.error
import urllib.parse
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

current_dir = Path.cwd().resolve()
# this is the currrent prohject path root
PROJECT_ROOT = next(
    folder for folder in [current_dir, *current_dir.parents]
    if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file()
)
# where noaa dataset will be stored
NOAA_FILE = PROJECT_ROOT / "data" / "raw" / "rdu_ghcnh_2015-01-01_to_2026-09-16.csv"
# where ecmwf dataset will be stored
ECMWF_FILE = PROJECT_ROOT / "data" / "raw" / "ecmwf_ifs_runs_2024-03-14_to_2026-09-16.csv"
# where gfs dataset will be stored (comparison only)
GFS_FILE = PROJECT_ROOT / "data" / "raw" / "gfs_seamless_runs_2026-04-02_to_2026-09-16.csv"

# Time zone, eastern america
NY = "America/New_York"
# Raleigh-Durham International Airport climate zhan
STATION_ID = "USW00013722"              
# RDU's latitude and longtitude
LATITUDE, LONGITUDE = 35.8922, -78.7819
# NOAA's dataset began downloading from this day.
NOAA_START = pd.Timestamp("2015-01-01", tz=NY)
CUTOFF = pd.Timestamp("2026-09-17", tz=NY)   # 12am Sep 17: nothing at or after this moment may be used
ECMWF_FIRST_RUN = pd.Timestamp("2024-03-14")  # first run in the Open-Meteo archive
ECMWF_LAST_RUN = pd.Timestamp("2026-09-16")
ECMWF_RUN_HOURS = (0, 12)                     # 00z and 12z runs (UTC)
GFS_FIRST_RUN = pd.Timestamp("2026-04-02")    # first 12z GFS run in the archive
GFS_LAST_RUN = pd.Timestamp("2026-09-16")

NOAA_URL = ("https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/access/"
            "by-year/{year}/psv/GHCNh_{station}_{year}.psv")
ECMWF_URL = "https://single-runs-api.open-meteo.com/v1/forecast"  # Open-Meteo Single Runs API (also used for GFS)
# 温度、露点、云量、风速、风向、降水、气压、太阳辐射
ECMWF_VARIABLES = ["temperature_2m", "dew_point_2m", "cloud_cover", "wind_speed_10m", "wind_direction_10m",
                   "precipitation", "pressure_msl", "shortwave_radiation"]

## 2. NOAA observations

- One file per UTC year is downloaded. Only routine hourly reports (`FM15`) taken at minute :51 are kept.
- Only reports from 12am Jan 1, 2015 up to (not including) 12am Sep 17, 2026 Eastern are kept, so the last one is 11:51pm Sep 16.
- Values stay as text, exactly as published; columns that are empty for the whole period are dropped. Cleaning happens in `02`.
- The reports are saved once per Eastern calendar year (`data/raw/2015/` … `data/raw/2026/`) and once as a single merged file.

In [15]:
frames = []
for year in tqdm(range(NOAA_START.year, CUTOFF.year + 1), desc="NOAA GHCNh", unit="year"):
    url = NOAA_URL.format(year=year, station=STATION_ID)
    frames.append(pd.read_csv(url, sep="|", dtype=str, keep_default_na=False))  # keep every value as text

noaa = pd.concat(frames, ignore_index=True).fillna("")
noaa = noaa[noaa["temperature_Report_Type"].eq("FM15") & noaa["Minute"].eq("51")]  # routine hourly reports
local_time = pd.to_datetime(noaa["DATE"], utc=True).dt.tz_convert(NY)
in_range = (local_time >= NOAA_START) & (local_time < CUTOFF)
noaa, local_time = noaa[in_range].copy(), local_time[in_range]
noaa.insert(noaa.columns.get_loc("DATE") + 1, "DATE_local", local_time.dt.strftime("%Y-%m-%d %H:%M"))
noaa = noaa.loc[:, noaa.ne("").any()]  # drop columns that are empty for the whole period

# One file per local (Eastern) year in data/raw/<year>/, plus one merged file
for year, rows in noaa.groupby(local_time.dt.year):
    first_day = max(NOAA_START, pd.Timestamp(f"{year}-01-01", tz=NY)).date()
    last_day = (min(CUTOFF, pd.Timestamp(f"{year + 1}-01-01", tz=NY)) - pd.Timedelta(days=1)).date()
    year_file = NOAA_FILE.parent / str(year) / f"rdu_ghcnh_{first_day}_to_{last_day}.csv"
    year_file.parent.mkdir(parents=True, exist_ok=True)
    rows.to_csv(year_file, index=False)
    print(f"Saved {year_file.relative_to(PROJECT_ROOT)}: {len(rows):,} rows")

noaa.to_csv(NOAA_FILE, index=False)
print(f"Saved {NOAA_FILE.relative_to(PROJECT_ROOT)}: {len(noaa):,} hourly reports x {noaa.shape[1]} columns, "
      f"{local_time.min():%Y-%m-%d %H:%M} -> {local_time.max():%Y-%m-%d %H:%M} Eastern")

NOAA GHCNh:   0%|          | 0/12 [00:00<?, ?year/s]

Saved data/raw/2015/rdu_ghcnh_2015-01-01_to_2015-12-31.csv: 8,760 rows
Saved data/raw/2016/rdu_ghcnh_2016-01-01_to_2016-12-31.csv: 8,781 rows
Saved data/raw/2017/rdu_ghcnh_2017-01-01_to_2017-12-31.csv: 8,757 rows
Saved data/raw/2018/rdu_ghcnh_2018-01-01_to_2018-12-31.csv: 8,757 rows
Saved data/raw/2019/rdu_ghcnh_2019-01-01_to_2019-12-31.csv: 8,759 rows
Saved data/raw/2020/rdu_ghcnh_2020-01-01_to_2020-12-31.csv: 8,784 rows
Saved data/raw/2021/rdu_ghcnh_2021-01-01_to_2021-12-31.csv: 8,756 rows
Saved data/raw/2022/rdu_ghcnh_2022-01-01_to_2022-12-31.csv: 8,760 rows
Saved data/raw/2023/rdu_ghcnh_2023-01-01_to_2023-12-31.csv: 8,760 rows
Saved data/raw/2024/rdu_ghcnh_2024-01-01_to_2024-12-31.csv: 8,784 rows
Saved data/raw/2025/rdu_ghcnh_2025-01-01_to_2025-12-31.csv: 8,641 rows
Saved data/raw/2026/rdu_ghcnh_2026-01-01_to_2026-09-16.csv: 6,172 rows
Saved data/raw/rdu_ghcnh_2015-01-01_to_2026-09-16.csv: 102,471 hourly reports x 226 columns, 2015-01-01 00:51 -> 2026-09-16 23:51 Eastern


## 3. ECMWF forecasts

- Each run is a 15-day hourly forecast for the model grid point nearest RDU (about 5 km away): 2 m temperature, dew point, cloud cover, wind, precipitation, pressure and solar radiation.
- The archive starts on Mar 14, 2024. Runs before mid-July 2024 only go out 10 days (240 h). A few runs are missing from the archive.
- The 00z and 12z runs of every day are downloaded. The real forecast uses the **12z run of Sep 16** (published that afternoon) together with older runs; nothing issued on Sep 17 or later is requested.
- Runs already in the file are skipped, so re-running this cell only fetches what is missing.

In [16]:
def fetch_run(run_time, model="ecmwf_ifs", retries=3):
    """One model run (360 forecast hours) at RDU -> (status, frame); status is ok / unavailable / failed."""
    query = urllib.parse.urlencode({
        "latitude": LATITUDE, "longitude": LONGITUDE, "models": model, "timezone": "GMT",
        "run": run_time.strftime("%Y-%m-%dT%H:%M"), "hourly": ",".join(ECMWF_VARIABLES), "forecast_hours": 360,
    })
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(f"{ECMWF_URL}?{query}", timeout=60) as response:
                hourly = json.load(response)["hourly"]
            break
        except urllib.error.HTTPError as error:
            if error.code == 400:  # "The requested model run is not available"
                return "unavailable", None
            if error.code == 429:  # API request limit reached: give up for now, re-run the cell later
                return "failed", None
            time.sleep(5 * 2 ** attempt)
        except (urllib.error.URLError, TimeoutError):
            time.sleep(5 * 2 ** attempt)
    else:
        return "failed", None
    run = pd.DataFrame(hourly).rename(columns={"time": "valid_utc"})
    run["valid_utc"] = pd.to_datetime(run["valid_utc"]).dt.tz_localize("UTC")
    run = run.dropna(subset=ECMWF_VARIABLES, how="all")
    run.insert(0, "run_utc", run_time)
    run.insert(2, "lead_hours", (run["valid_utc"] - run_time) // pd.Timedelta(hours=1))
    return "ok", run

def download_runs(model, runs, file):
    """Request the runs that are not in `file` yet, add them to the file and return the whole table."""
    existing = pd.read_csv(file, parse_dates=["run_utc", "valid_utc"]) if file.exists() else None
    done = set(existing["run_utc"]) if existing is not None else set()
    todo = [run for run in runs if run not in done]
    print(f"Runs already in {file.relative_to(PROJECT_ROOT)}: {len(done):,}; to request: {len(todo):,}")

    with ThreadPoolExecutor(max_workers=8) as pool:  # about 1,800 small requests for ECMWF on a first run
        results = list(tqdm(pool.map(lambda run: fetch_run(run, model), todo), total=len(todo), desc=model, unit="run"))

    new_runs = [frame for status, frame in results if status == "ok"]
    unavailable = [run for run, (status, _) in zip(todo, results) if status == "unavailable"]
    failed = [run for run, (status, _) in zip(todo, results) if status == "failed"]
    table = pd.concat([existing] + new_runs, ignore_index=True) if new_runs else existing
    if table is None:
        raise RuntimeError(f"No {model} data: {file.relative_to(PROJECT_ROOT)} does not exist and nothing could be "
                           "downloaded. Check the file path in the settings cell, or try again later.")
    if new_runs:
        table = table.sort_values(["run_utc", "valid_utc"]).reset_index(drop=True)
        file.parent.mkdir(parents=True, exist_ok=True)
        table.to_csv(file, index=False, date_format="%Y-%m-%dT%H:%MZ")
    print(f"{file.relative_to(PROJECT_ROOT)}: {table['run_utc'].nunique():,} runs, {len(table):,} rows "
          f"({len(new_runs)} new)")
    print(f"Not in the archive ({len(unavailable)}): {[f'{r:%Y-%m-%d %H}z' for r in unavailable]}")
    if failed:
        print(f"{len(failed)} runs could not be downloaded now (e.g. API request limit) - run this cell again later")
    return table


all_runs = [day.tz_localize("UTC") + pd.Timedelta(hours=hour)
            for day in pd.date_range(ECMWF_FIRST_RUN, ECMWF_LAST_RUN, freq="D") for hour in ECMWF_RUN_HOURS]
ecmwf = download_runs("ecmwf_ifs", all_runs, ECMWF_FILE)

Runs already in data/raw/ecmwf_ifs_runs_2024-03-14_to_2026-09-16.csv: 1,828; to request: 6


ecmwf_ifs:   0%|          | 0/6 [00:00<?, ?run/s]

data/raw/ecmwf_ifs_runs_2024-03-14_to_2026-09-16.csv: 1,828 runs, 628,281 rows (0 new)
Not in the archive (6): ['2025-08-05 00z', '2025-08-06 00z', '2025-08-08 00z', '2025-08-08 12z', '2025-08-09 00z', '2026-06-10 12z']


## 4. GFS forecasts (comparison only)

- GFS is the global weather model of the US National Weather Service, the American counterpart of ECMWF. It comes from the same API (`gfs_seamless`), with the same variables and 360 forecast hours.
- It is **not a model input**. It is only a second raw forecast, compared with raw ECMWF in `04` and `05`, to check whether ECMWF is the better forecast to build on.
- The archive of 12z GFS runs starts on Apr 2, 2026, so GFS can only be compared on the 2026 validation fold and the test window. The Sep 14, 2026 run is not in the archive.
- Same rules as ECMWF: only runs started before the cutoff, and runs already in the file are skipped.

In [17]:
gfs_runs = [day.tz_localize("UTC") + pd.Timedelta(hours=12) for day in pd.date_range(GFS_FIRST_RUN, GFS_LAST_RUN, freq="D")]
gfs = download_runs("gfs_seamless", gfs_runs, GFS_FILE)

Runs already in data/raw/gfs_seamless_runs_2026-04-02_to_2026-09-16.csv: 167; to request: 1


gfs_seamless:   0%|          | 0/1 [00:00<?, ?run/s]

data/raw/gfs_seamless_runs_2026-04-02_to_2026-09-16.csv: 167 runs, 60,120 rows (0 new)
Not in the archive (1): ['2026-09-14 12z']


## 5. Checks

- No observation at or after the cutoff, and the Sep 16 12z runs of ECMWF and GFS cover the whole target window.
- Time alignment: a report taken at hh:51 is compared with the forecast valid at hh:00, hh+1:00 and hh+2:00 (UTC). The smallest difference at +1 h confirms the matching used later: the report labelled hh ↔ the forecast valid at hh+1:00.

In [18]:
obs_utc = pd.to_datetime(noaa["DATE"], utc=True)
run_length = ecmwf.groupby("run_utc")["lead_hours"].max()
gfs_length = gfs.groupby("run_utc")["lead_hours"].max()
sep16_12z = pd.Timestamp("2026-09-16 12:00", tz="UTC")
checks = {
    "NOAA: one report per hour, all at minute :51": obs_utc.is_unique and noaa["Minute"].eq("51").all(),
    "NOAA: nothing at/after 12am Sep 17": (obs_utc < CUTOFF).all(),
    "ECMWF: every run starts before the cutoff": (run_length.index < CUTOFF).all(),
    "ECMWF: the Sep 16 12z run covers Sep 17-30": run_length.get(sep16_12z, 0) >= 352,
    "GFS: every run starts before the cutoff": (gfs_length.index < CUTOFF).all(),
    "GFS: the Sep 16 12z run covers Sep 17-30": gfs_length.get(sep16_12z, 0) >= 352,
}
display(pd.Series(checks, name="passed").to_frame())
assert all(checks.values())
print("ECMWF runs by forecast length (hours):", run_length.value_counts().sort_index().to_dict())

# Time alignment: a report at hh:51 is closest to the forecast valid at hh+1:00 -> the error should be smallest at +1 h
temperature = pd.Series(pd.to_numeric(noaa["temperature"]).to_numpy(), index=obs_utc.dt.floor("h"))
short = ecmwf[ecmwf["lead_hours"].between(6, 48)]
for offset in (0, 1, 2):
    observed = temperature.reindex(short["valid_utc"] - pd.Timedelta(hours=offset)).to_numpy()
    print(f"report hour + {offset} h: mean absolute difference {np.nanmean(np.abs(observed - short['temperature_2m'])):.2f} deg C")

,passed
"NOAA: one report per hour, all at minute :51",True
NOAA: nothing at/after 12am Sep 17,True
ECMWF: every run starts before the cutoff,True
ECMWF: the Sep 16 12z run covers Sep 17-30,True
GFS: every run starts before the cutoff,True
GFS: the Sep 16 12z run covers Sep 17-30,True


ECMWF runs by forecast length (hours): {72: 1, 240: 248, 359: 1579}
report hour + 0 h: mean absolute difference 1.62 deg C
report hour + 1 h: mean absolute difference 1.55 deg C
report hour + 2 h: mean absolute difference 1.86 deg C
